<a href="https://colab.research.google.com/github/huggingface/deep-rl-class/blob/main/notebooks/unit5/unit5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第 5 单元:ML-Agents 入门



<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/thumbnail.png" alt="Thumbnail"/>

在本 notebook 中,你将了解 ML-Agents 并训练两个智能体。

- 第一个智能体将学会**向不断出现的目标投掷雪球**。
- 第二个智能体需要按下按钮生成一座金字塔,然后走到金字塔那里把它推倒,**再移动到顶部的金砖处**。要做到这一点,它需要探索它的环境,我们将使用一种叫做好奇心(curiosity)的技术。

之后,你就可以**直接在浏览器中观看你的智能体游玩**了。

有关认证流程的更多信息,请查看这一节 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process


⬇️ 下面是**你在本单元结束时将实现的效果**示例 ⬇️


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/pyramids.gif" alt="Pyramids"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget.gif" alt="SnowballTarget"/>

### 🎮 环境:

- [Pyramids](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Learning-Environment-Examples.md#pyramids)
- SnowballTarget

### 📚 强化学习库:

- [ML-Agents](https://github.com/Unity-Technologies/ml-agents)


我们一直在努力改进我们的教程,因此**如果你在本 notebook 中发现一些问题**,请在 [GitHub 仓库上提一个 issue](https://github.com/huggingface/deep-rl-class/issues)。


## 本 notebook 的学习目标 🏆

完成本 notebook 后,你将:

- 理解 **ML-Agents** 这个环境库是如何工作的。
- 能够**在 Unity 环境中训练智能体**。


## 本 notebook 来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>


在这个免费课程中,你将:

- 📖 从**理论和实践**两方面学习深度强化学习。
- 🧑‍💻 学会**使用知名的深度强化学习库**,例如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在独特的环境中**训练智能体**

更多内容请查看 📚 课程大纲 👉 https://huggingface.co/deep-rl-course/communication/publishing-schedule

别忘了**<a href="http://eepurl.com/ic5ZUD">注册课程</a>**(我们收集你的邮箱,是为了**在每个单元发布时把链接发送给你,并向你提供有关挑战和更新的信息**)。


保持联系最好的方式是加入我们的 Discord 服务器,与社区以及我们交流 👉🏻 https://discord.gg/ydHrjt3WP5


## 前置知识 🏗️
在开始本 notebook 之前,你需要:

🔲 📚 **通过阅读第 5 单元学习[什么是 ML-Agents 以及它是如何工作的](https://huggingface.co/deep-rl-course/unit5/introduction)**  🤗  


# 我们来训练我们的智能体吧 🚀

**为了在认证流程中通过本次动手实践,你只需要把训练好的模型推送到 Hub 即可**。这次实践没有必须达到的成绩要求。但如果你想获得漂亮的成绩,可以试着达到:

- `Pyramids`:平均奖励 = 1.75
- `SnowballTarget`:平均奖励 = 15,即一个回合中击中 30 个目标。


## 设置 GPU 💪
- 为了**加速智能体的训练,我们将使用 GPU**。为此,前往 `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">


- `硬件加速器(Hardware Accelerator)> GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">


## 克隆仓库 🔽

- 我们需要克隆包含 **ML-Agents** 的仓库。


In [ ]:
%%capture
# 克隆仓库(可能需要 3 分钟)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## 配置虚拟环境 🔽
- 为了让 **ML-Agents** 能够在 Colab 中成功运行,Colab 的 Python 版本必须满足该库的 Python 版本要求。

- 我们可以在 `setup.py` 文件的 `python_requires` 参数下查看受支持的 Python 版本。这些文件是安装和使用 **ML-Agents** 库所必需的,位于以下位置:
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab 当前的 Python 版本(可以用 `!python --version` 查看)与该库的 `python_requires` 参数不匹配,因此安装可能会静默失败,并在之后执行相同命令时导致如下错误:
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- 为了解决这个问题,我们将创建一个 Python 版本与 **ML-Agents** 库兼容的虚拟环境。

`注意:` *为了未来的兼容性,请务必检查安装文件中的 `python_requires` 参数;如果 Colab 的 Python 版本不兼容,请在下面给出的脚本中把你的虚拟环境设置为所支持的最高 Python 版本*


In [ ]:
# Colab 当前的 Python 版本(与 ML-Agents 不兼容)
!python --version

In [ ]:
# 安装 virtualenv 并创建虚拟环境
!pip install virtualenv
!virtualenv myenv

# 下载并安装 Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# 激活 Miniconda 并安装 Python 3.10.12 版本
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # 在这里指定版本

# 为 Python 和 conda 路径设置环境变量
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv

In [ ]:
# 新虚拟环境中的 Python 版本(与 ML-Agents 兼容)
!python --version

## 安装依赖 🔽


In [ ]:
%%capture
# 进入仓库并安装软件包(可能需要 3 分钟)
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

## SnowballTarget ⛄

如果你想回顾一下这个环境是如何工作的,请查看这一节 👉
https://huggingface.co/deep-rl-course/unit5/snowball-target


### 下载环境 zip 文件并移动到 `./training-envs-executables/linux/`
- 我们的环境可执行文件在一个 zip 文件中。
- 我们需要下载它并放到 `./training-envs-executables/linux/`
- 我们使用 Linux 可执行文件,因为我们使用的是 Colab,而 Colab 机器的操作系统是 Ubuntu(Linux)


In [ ]:
# 在这里,我们创建 training-envs-executables 和 linux 目录
!mkdir ./training-envs-executables
!mkdir ./training-envs-executables/linux

我们使用 `wget` 从 https://github.com/huggingface/Snowball-Target 下载了 SnowballTarget.zip 文件


In [ ]:
!wget "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" -O ./training-envs-executables/linux/SnowballTarget.zip

我们解压 executable.zip 文件


In [ ]:
%%capture
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/SnowballTarget.zip

确保你的文件是可访问的


In [ ]:
!chmod -R 755 ./training-envs-executables/linux/SnowballTarget

### 定义 SnowballTarget 配置文件
- 在 ML-Agents 中,你要把**训练超参数定义在 config.yaml 文件中**。

超参数有很多个。为了更好地了解它们,你应该到[文档](https://github.com/Unity-Technologies/ml-agents/blob/release_20_docs/docs/Training-Configuration-File.md)中查看每个超参数的说明。


所以你需要在 ./content/ml-agents/config/ppo/ 中创建一个 `SnowballTarget.yaml` 配置文件。

我们在这里给出该配置的第一个版本(可复制粘贴到你的 `SnowballTarget.yaml 文件` 中),**但你应该对它进行修改**。

```
behaviors:
  SnowballTarget:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 64
    threaded: false
    hyperparameters:
      learning_rate: 0.0003
      learning_rate_schedule: linear
      batch_size: 128
      buffer_size: 2048
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
```


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config1.png" alt="Config SnowballTarget"/>
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config2.png" alt="Config SnowballTarget"/>

作为实验,你也应该尝试修改其他一些超参数,Unity 在[这里](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)提供了非常好的文档,解释了每个超参数的作用。

既然你已经创建了配置文件,并理解了大多数超参数的作用,我们就可以开始训练我们的智能体了 🔥。


### 训练智能体

要训练我们的智能体,我们只需要**启动 mlagents-learn 并选择包含环境的可执行文件。**

我们定义四个参数:

1. `mlagents-learn <config>`:超参数配置文件所在的路径。
2. `--env`:环境可执行文件所在的位置。
3. `--run_id`:你想给本次训练 run id 起的名称。
4. `--no-graphics`:训练期间不启动可视化。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/mlagentslearn.png" alt="MlAgents learn"/>

训练模型时,使用 `--resume` 标志可以在训练中断后继续训练。

> 当你使用 `--resume` 时,第一次可能会失败,再次运行该代码块即可绕过这个错误。



训练将花费 10 到 35 分钟,取决于你的配置。去喝杯 ☕️ 吧,这是你应得的 🤗。


In [ ]:
!mlagents-learn ./config/ppo/SnowballTarget.yaml --env=./training-envs-executables/linux/SnowballTarget/SnowballTarget --run-id="SnowballTarget1" --no-graphics

### 把智能体推送到 🤗 Hub

- 既然我们已经训练好了智能体,就**可以把它推送到 Hub,以便在浏览器中观看它游玩 🔥**。


要想与社区分享你的模型,还需要再完成三个步骤:

1️⃣ (如果还没有的话)注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录之后,你需要保存来自 Hugging Face 网站的认证令牌(token)。
- 创建一个新令牌(https://huggingface.co/settings/tokens),**角色为写入(write)权限**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制该令牌
- 运行下面的单元格并粘贴令牌


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果你不想使用 Google Colab 或 Jupyter Notebook,则需要改用这个命令:`huggingface-cli login`


然后,我们只需要运行 `mlagents-push-to-hf`。

我们定义 4 个参数:

1. `--run-id`:训练 run id 的名称。
2. `--local-dir`:智能体保存的位置,即 results/<run_id 名称>,所以在我的例子中是 results/First Training。
3. `--repo-id`:你要创建或更新的 Hugging Face 仓库名称。它始终是 <你的 Hugging Face 用户名>/<仓库名称>。
如果该仓库不存在,**它会被自动创建**
4. `--commit-message`:由于 HF 仓库是 git 仓库,你需要定义一条提交信息(commit message)。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/mlagentspushtohub.png" alt="Push to Hub"/>

例如:

`!mlagents-push-to-hf  --run-id="SnowballTarget1" --local-dir="./results/SnowballTarget1" --repo-id="ThomasSimonini/ppo-SnowballTarget"  --commit-message="First Push"`


In [ ]:
!mlagents-push-to-hf --run-id="SnowballTarget1" --local-dir="./results/SnowballTarget1" --repo-id="ThomasSimonini/ppo-SnowballTarget" --commit-message="First Push"

In [ ]:
!mlagents-push-to-hf  --run-id= # 填入你的 run id  --local-dir= # 你的本地目录  --repo-id= # 你的 repo id  --commit-message= # 你的提交信息

另外,如果一切顺利,你应该会在流程结束时看到如下内容(但 URL 不同 😆):



```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-SnowballTarget
```

这是你的模型的链接,它包含一张解释如何使用它的模型卡片、你的 Tensorboard 以及你的配置文件。**最棒的是它是一个 git 仓库,这意味着你可以拥有不同的提交,通过新的推送来更新你的仓库等等。**


但现在迎来了最精彩的部分:**能够在线观看你的智能体游玩 👀**。


### 观看你的智能体游玩 👀

这一步很简单:

1. 前往这里:https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget

2. 启动游戏,并点击右下角的按钮将其全屏显示

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget_load.png" alt="Snowballtarget load"/>


1. 在第 1 步中,输入你的用户名(用户名区分大小写:例如,我的用户名是 ThomasSimonini,而不是 thomassimonini 或 ThOmasImoNInI),然后点击搜索按钮。

2. 在第 2 步中,选择你的模型仓库。

3. 在第 3 步中,**选择你想回放哪个模型**:
  - 我有多个可选,因为我们每 500000 个时间步就保存一次模型。
  - 但由于我想要最新的一个,所以我选择 `SnowballTarget.onnx`

👉 很棒的一点是**可以逐步尝试不同的模型,以观察智能体的进步。**

欢迎在 Discord 的 #rl-i-made-this 频道上分享你的智能体取得的最高分 🔥

现在让我们来尝试一个更难的环境,叫做 Pyramids……


## Pyramids 🏆

### 下载环境 zip 文件并移动到 `./training-envs-executables/linux/`
- 我们的环境可执行文件在一个 zip 文件中。
- 我们需要下载它并放到 `./training-envs-executables/linux/`
- 我们使用 Linux 可执行文件,因为我们使用的是 Colab,而 Colab 机器的操作系统是 Ubuntu(Linux)


我们使用 `wget` 从 https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip 下载了 Pyramids.zip 文件


In [ ]:
!wget "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" -O ./training-envs-executables/linux/Pyramids.zip

我们解压 executable.zip 文件


In [ ]:
%%capture
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/Pyramids.zip

确保你的文件是可访问的


In [ ]:
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids

###  修改 PyramidsRND 配置文件
- 与第一个自定义环境不同,**Pyramids 是由 Unity 团队制作的**。
- 因此 PyramidsRND 配置文件已经存在,位于 ./content/ml-agents/config/ppo/PyramidsRND.yaml
- 你可能会问 PyramidsRND 中的 "RND" 是什么意思。RND 代表*随机网络蒸馏(random network distillation)*,它是一种生成好奇心奖励的方法。如果你想了解更多,我们写了一篇文章来解释这一技术:https://medium.com/data-from-the-trenches/curiosity-driven-learning-through-random-network-distillation-488ffd8e5938

对于这次训练,我们将修改一处:
- 总训练步数这个超参数太高了,因为我们只需 1M 训练步就能达到基准(平均奖励 = 1.75)。
👉 为此,我们打开 config/ppo/PyramidsRND.yaml,**把 max_steps 修改为 1000000。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/pyramids-config.png" alt="Pyramids config"/>


作为实验,你也应该尝试修改其他一些超参数,Unity 在[这里](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)提供了非常好的文档,解释了每个超参数的作用。

我们现在已经准备好训练我们的智能体了 🔥。


### 训练智能体

训练将花费 30 到 45 分钟,取决于你的机器。去喝杯 ☕️ 吧,这是你应得的 🤗。


In [ ]:
!mlagents-learn ./config/ppo/PyramidsRND.yaml --env=./training-envs-executables/linux/Pyramids/Pyramids --run-id="Pyramids Training" --no-graphics

### 把智能体推送到 🤗 Hub

- 既然我们已经训练好了智能体,就**可以把它推送到 Hub,以便在浏览器中观看它游玩 🔥**。


In [ ]:
!mlagents-push-to-hf  --run-id= # 填入你的 run id  --local-dir= # 你的本地目录  --repo-id= # 你的 repo id  --commit-message= # 你的提交信息

### 观看你的智能体游玩 👀

👉 https://huggingface.co/spaces/unity/ML-Agents-Pyramids


### 🎁 加餐:为什么不在另一个环境上训练呢?
既然你已经知道如何使用 MLAgents 训练智能体,**为什么不试试其他环境呢?**

MLAgents 提供了 17 个不同的环境,我们还在构建一些自定义环境。学习的最好方式就是自己动手尝试,祝你好玩。



![cover](https://miro.medium.com/max/1400/0*xERdThTRRM2k_U9f.png)

Unity 官方环境的完整列表在这里 👉 https://github.com/Unity-Technologies/ml-agents/blob/develop/docs/Learning-Environment-Examples.md

用于观看你智能体演示的 Demo 在这里 👉 https://huggingface.co/unity

目前我们已经集成了:
- [Worm](https://huggingface.co/spaces/unity/ML-Agents-Worm) 演示,你可以教一条**虫子爬行**。
- [Walker](https://huggingface.co/spaces/unity/ML-Agents-Walker) 演示,你可以教一个智能体**朝目标行走**。


今天的内容就到这里。恭喜你完成了本教程!

学习的最好方式就是动手实践、大胆尝试。为什么不试试另一个环境呢?ML-Agents 有 17 个不同的环境,你也可以创建自己的环境。查阅文档,玩得开心!

第 6 单元见 🔥,

## 持续学习,保持出色 🤗
